# 🏙️ City Human Sim — Training v4 (フルGPU版)
**Google Colab / A100対応**

### v4の改善: 環境ステップも全てGPU上で実行
- **環境ステップをtorch化** → CPU↔GPU転送ゼロ
- **N_ENVS=4096** → A100の80GBをフル活用
- CPU待機がなくなりGPU使用率90%+を目指す


## Step 1: インストール

In [ ]:
import subprocess, sys
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])
pip("torch", "torchvision")
pip("onnx", "onnxruntime")
pip("numpy", "matplotlib")
print("✓ Done")


## Step 2: GPU確認

In [ ]:
import torch
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    print(f"✓ GPU: {props.name}")
    print(f"  Memory: {props.total_memory/1e9:.1f} GB")
    print(f"  SM count: {props.multi_processor_count}")
else:
    DEVICE = torch.device("cpu")
    print("⚠ CPU mode")
print(f"\nDEVICE = {DEVICE}")


## Step 3: 設定

In [ ]:
GRID_SIZE    = 10
N_AGENTS     = 6
MAX_STEPS    = 300
N_UPDATES    = 3000
N_ENVS       = 4096   # A100 80GB → 大きく取れる
CELL         = 1.0

LR           = 3e-4
GAMMA        = 0.99
ENTROPY_COEF = 0.05

REWARD_NEW_CELL = 3.0
REWARD_MEETING  = 1.5
PENALTY_STAY    = 0.3

# 移動ベクトル (GPU tensor として保持)
import torch
import torch.nn as nn
import numpy as np

MOVES_T = torch.tensor(
    [[0,1],[0,-1],[-1,0],[1,0],[0,0]], dtype=torch.float32
)  # (5, 2)

print(f"✓ N_ENVS={N_ENVS}, N_UPDATES={N_UPDATES}")
print(f"  総ステップ数: {N_ENVS * MAX_STEPS * N_UPDATES:,}")


## Step 4: モデル・フルGPU環境定義

In [ ]:
# ============================================================
# Actor-Critic
# ============================================================
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(6, 256), nn.LayerNorm(256), nn.ReLU(),
            nn.Linear(256, 256), nn.LayerNorm(256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.actor  = nn.Linear(128, 5)
        self.critic = nn.Linear(128, 1)

    def forward(self, x):
        h = self.encoder(x)
        return self.actor(h), self.critic(h).squeeze(-1)

    def act_batch(self, obs):
        logits, values = self(obs)
        dist = torch.distributions.Categorical(logits=logits)
        actions = dist.sample()
        return actions, dist.log_prob(actions), dist.entropy(), values


# ============================================================
# フルGPU並列環境
# 全テンソルをGPU上で保持・計算 → CPU転送ゼロ
# ============================================================
class GPUCitySimulator:
    def __init__(self, n_envs, device):
        self.n   = n_envs
        self.dev = device
        self.moves = MOVES_T.to(device)  # (5,2) GPU上
        self.reset()

    def reset(self):
        G, N, D = GRID_SIZE, self.n, self.dev

        # エージェント位置: (n_envs, n_agents, 2)
        self.pos = torch.randint(0, G, (N, N_AGENTS, 2),
                                 dtype=torch.float32, device=D)

        # 訪問フラグ: (n_envs, G, G) bool
        self.visited   = torch.zeros(N, G, G, dtype=torch.bool, device=D)
        # 訪問頻度: (n_envs, G, G) float
        self.visit_cnt = torch.zeros(N, G, G, dtype=torch.float32, device=D)
        # 累積出会い数: (n_envs,)
        self.meetings  = torch.zeros(N, dtype=torch.float32, device=D)
        # ステップカウント: (n_envs,)
        self.steps     = torch.zeros(N, dtype=torch.int32, device=D)

        # 初期位置をvisited登録
        x0 = self.pos[:, 0, 0].long()
        y0 = self.pos[:, 0, 1].long()
        self.visited[torch.arange(N, device=D), x0, y0] = True

        return self._obs()

    def _obs(self):
        """観測ベクトル (n_envs, 6) を全てGPU上で計算"""
        N, G, D = self.n, GRID_SIZE, self.dev
        pos    = self.pos[:, 0, :]          # (N, 2) 主エージェント
        others = self.pos[:, 1:, :]         # (N, n_agents-1, 2)

        diffs  = others - pos.unsqueeze(1)  # (N, n_agents-1, 2)
        dists  = diffs.norm(dim=-1)         # (N, n_agents-1)
        ni     = dists.argmin(dim=-1)       # (N,)
        nearest = diffs[torch.arange(N, device=D), ni]  # (N, 2)

        visited_ratio = self.visited.float().sum(dim=(1,2)) / (G * G)  # (N,)
        meeting_rate  = (self.meetings / 20.0).clamp(0, 1)             # (N,)

        obs = torch.stack([
            pos[:, 0] / G,
            pos[:, 1] / G,
            visited_ratio,
            meeting_rate,
            nearest[:, 0] / G,
            nearest[:, 1] / G,
        ], dim=-1)  # (N, 6)
        return obs.clamp(-1.0, 1.0)

    def step(self, actions):
        """
        actions: (N,) LongTensor on GPU
        全計算をGPU上で実行 → CPU転送ゼロ
        """
        N, G, D = self.n, GRID_SIZE, self.dev
        idx = torch.arange(N, device=D)

        # 主エージェント移動
        delta   = self.moves[actions]                      # (N, 2)
        new_pos = (self.pos[:, 0, :] + delta).clamp(0, G-1)
        self.pos[:, 0, :] = new_pos

        # 他エージェントランダム移動
        rand_a = torch.randint(0, 5, (N, N_AGENTS-1), device=D)
        for a in range(N_AGENTS - 1):
            rd = self.moves[rand_a[:, a]]
            self.pos[:, a+1, :] = (self.pos[:, a+1, :] + rd).clamp(0, G-1)

        self.steps += 1

        # ── 報酬計算（全GPU） ──
        cx = new_pos[:, 0].long()
        cy = new_pos[:, 1].long()

        # 新規セル訪問ボーナス
        new_cell = ~self.visited[idx, cx, cy]
        rewards  = new_cell.float() * REWARD_NEW_CELL
        self.visited[idx, cx, cy] = True

        # 好奇心ボーナス
        curiosity = 1.0 / (1.0 + self.visit_cnt[idx, cx, cy])
        rewards  += curiosity * 0.5
        self.visit_cnt[idx, cx, cy] += 1

        # 出会いボーナス
        others = self.pos[:, 1:, :]                         # (N, n_agents-1, 2)
        dists  = (others - new_pos.unsqueeze(1)).norm(dim=-1)  # (N, n_agents-1)
        n_meet = (dists < 1.5).float().sum(dim=-1)          # (N,)
        rewards += n_meet * REWARD_MEETING
        self.meetings += n_meet

        # 停止ペナルティ
        rewards -= (actions == 4).float() * PENALTY_STAY

        dones = self.steps >= MAX_STEPS
        return self._obs(), rewards, dones

    def reset_done(self, dones):
        """終了環境だけリセット（GPU上）"""
        if not dones.any():
            return
        N, G, D = self.n, GRID_SIZE, self.dev
        di = dones.nonzero(as_tuple=True)[0]
        n_done = len(di)

        self.pos[di]       = torch.randint(0, G, (n_done, N_AGENTS, 2),
                                           dtype=torch.float32, device=D)
        self.visited[di]   = False
        self.visit_cnt[di] = 0
        self.meetings[di]  = 0
        self.steps[di]     = 0

        x0 = self.pos[di, 0, 0].long()
        y0 = self.pos[di, 0, 1].long()
        self.visited[di, x0, y0] = True

print("✓ ActorCritic, GPUCitySimulator defined")
print(f"  推論・環境ステップ・報酬計算 → 全てGPU上で実行")


## Step 5: 学習実行

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import clear_output
import time

matplotlib.rcParams['figure.dpi'] = 90

def train():
    print("=" * 60)
    print(f"Full-GPU Actor-Critic: {N_ENVS} envs × {N_UPDATES} updates")
    print(f"Device: {DEVICE}")
    print("=" * 60)

    model     = ActorCritic().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=N_UPDATES, eta_min=LR * 0.05
    )
    envs = GPUCitySimulator(N_ENVS, DEVICE)
    obs  = envs.reset()  # (N_ENVS, 6) on GPU

    ep_rewards  = []
    ep_visited  = []
    ep_meetings = []
    ep_entropy  = []

    start_time = time.time()

    for update in range(N_UPDATES):
        all_log_probs = []
        all_values    = []
        all_rewards   = []
        all_entropies = []
        envs._ep_visited_buf  = []
        envs._ep_meetings_buf = []

        # ── ロールアウト収集（全GPU） ──
        for _ in range(MAX_STEPS):
            actions, lps, ents, vals = model.act_batch(obs)
            obs, rewards, dones = envs.step(actions)

            all_log_probs.append(lps)
            all_values.append(vals)
            all_rewards.append(rewards)
            all_entropies.append(ents)
            # doneになった環境の終端統計を保存（resetする前に記録）
            if dones.any():
                _v = envs.visited.float().sum(dim=(1,2))  # (N,)
                _m = envs.meetings                         # (N,)
                if not hasattr(envs, '_ep_visited_buf'):
                    envs._ep_visited_buf  = []
                    envs._ep_meetings_buf = []
                envs._ep_visited_buf.extend(_v[dones].tolist())
                envs._ep_meetings_buf.extend(_m[dones].tolist())
            envs.reset_done(dones)

        # ── 勾配更新 ──
        log_probs_t = torch.stack(all_log_probs)   # (T, N)
        values_t    = torch.stack(all_values)       # (T, N)
        rewards_t   = torch.stack(all_rewards)      # (T, N)
        entropies_t = torch.stack(all_entropies)    # (T, N)

        # リターン計算
        returns = torch.zeros_like(rewards_t)
        G = torch.zeros(N_ENVS, device=DEVICE)
        for t in reversed(range(MAX_STEPS)):
            G = rewards_t[t] + GAMMA * G
            returns[t] = G

        adv = returns - values_t.detach()
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        actor_loss   = -(log_probs_t * adv).mean()
        critic_loss  = nn.functional.mse_loss(values_t, returns.detach())
        entropy_loss = -entropies_t.mean()
        loss = actor_loss + 0.5 * critic_loss + ENTROPY_COEF * entropy_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        scheduler.step()

        # 終端統計: バッファが空なら現在値をフォールバック
        if len(envs._ep_visited_buf) > 0:
            visited_at_end  = torch.tensor(envs._ep_visited_buf, device=DEVICE)
            meetings_at_end = torch.tensor(envs._ep_meetings_buf, device=DEVICE)
        else:
            visited_at_end  = envs.visited.float().sum(dim=(1,2))
            meetings_at_end = envs.meetings

        # 統計: ロールアウト中に記録した値を使う（reset_done後は0になるため）
        ep_rewards.append(rewards_t.sum(dim=0).mean().item())   # T×N→各env合計→平均
        ep_visited.append(visited_at_end.mean().item())          # ロールアウト末尾時点
        ep_meetings.append(meetings_at_end.mean().item())        # 同上
        ep_entropy.append(entropies_t.mean().item())

        if (update + 1) % 300 == 0:
            elapsed = time.time() - start_time
            sps = (update + 1) * N_ENVS * MAX_STEPS / elapsed

            mem_used  = torch.cuda.memory_allocated(0) / 1e9
            mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9

            clear_output(wait=True)
            fig, axes = plt.subplots(2, 2, figsize=(14, 8))
            fig.suptitle(
                f"Update {update+1}/{N_UPDATES}  |  {sps/1e6:.2f}M steps/sec  |"
                f"  GPU Mem: {mem_used:.1f}/{mem_total:.1f} GB",
                fontsize=11, color="white"
            )

            def plot(ax, data, title, color, ylabel, hline=None):
                w = 100
                s = [np.mean(data[max(0,i-w):i+1]) for i in range(len(data))]
                ax.plot(data, alpha=0.2, color=color, lw=0.6)
                ax.plot(s, color=color, lw=2)
                if hline:
                    ax.axhline(hline, color="white", ls="--", alpha=0.3,
                               label=f"Max={hline}")
                    ax.legend(labelcolor="white", fontsize=8)
                ax.set_title(title, color="white")
                ax.set_ylabel(ylabel); ax.set_xlabel("Update")
                ax.set_facecolor("#1a1a2e"); ax.grid(alpha=0.2)
                for item in ([ax.xaxis.label, ax.yaxis.label]
                             + ax.get_xticklabels() + ax.get_yticklabels()):
                    item.set_color("#aaaacc")
                for sp in ax.spines.values():
                    sp.set_edgecolor("#334")

            plot(axes[0,0], ep_rewards,  "Avg Reward",        "#4a9eff", "Reward")
            plot(axes[0,1], ep_visited,  "Avg Cells Visited", "#44ff88", "Cells",
                 hline=GRID_SIZE**2)
            plot(axes[1,0], ep_meetings, "Avg Meetings",      "#ff6688", "Meetings")
            plot(axes[1,1], ep_entropy,  "Entropy (探索度)",   "#ffcc44", "Entropy")
            fig.patch.set_facecolor("#0f0f1a")
            plt.tight_layout()
            plt.show()

            print(f"Update {update+1:5d} | "
                  f"Reward {np.mean(ep_rewards[-100:]):6.1f} | "
                  f"Visited {ep_visited[-1]:.1f}/{GRID_SIZE**2} | "
                  f"Entropy {ep_entropy[-1]:.3f} | "
                  f"{sps/1e6:.2f}M steps/sec | "
                  f"GPU {mem_used:.1f}/{mem_total:.1f}GB")

    print(f"\n✓ Done! Total: {N_UPDATES*N_ENVS*MAX_STEPS/1e9:.2f}B steps")
    return model

model = train()


## Step 6: ONNX エクスポート

In [ ]:
import json

class ActorOnly(nn.Module):
    def __init__(self, ac):
        super().__init__()
        self.encoder = ac.encoder
        self.actor   = ac.actor
    def forward(self, x):
        return self.actor(self.encoder(x))

def export_onnx(model, path="policy.onnx"):
    actor = ActorOnly(model).to("cpu").eval()
    dummy = torch.zeros(1, 6)
    torch.onnx.export(
        actor, dummy, path,
        input_names=["state"], output_names=["logits"],
        dynamic_axes={"state":{0:"batch"},"logits":{0:"batch"}},
        opset_version=11,
    )
    print(f"✓ ONNX saved: {path}")
    meta = {
        "input_size":6, "output_size":5,
        "actions":["up","down","left","right","stay"],
        "obs_keys":["pos_x/GRID","pos_y/GRID","visited_ratio",
                    "meeting_rate","nearest_dx/GRID","nearest_dy/GRID"],
        "grid_size":GRID_SIZE, "n_agents":N_AGENTS, "cell_size":CELL,
        "model_version":"v4_full_gpu"
    }
    with open(path.replace(".onnx","_meta.json"),"w") as f:
        json.dump(meta, f, indent=2)
    print("✓ Metadata saved")

export_onnx(model)


## Step 7: ダウンロード

In [ ]:
from google.colab import files
files.download("policy.onnx")
files.download("policy_meta.json")
print("✓ city_sim.html に policy.onnx を読み込んでください")
